# MOSS-Image on a free Colab T4
Text-to-image with [MOSS-Image-SFT](https://huggingface.co/OpenMOSS-Team/MOSS-Image-SFT) (5B parameters, Apache-2.0, strong at text inside images). Its weights are 27 GB, so this notebook stores the image model as **int8** and the text encoder as **4-bit** to fit a free T4 (16 GB).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The first run downloads about 25 GB (around 15 minutes), then loading takes about 2.5 minutes and each 768×768 picture about 2 minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Install
`optimum-quanto` stores the image model in int8 and `bitsandbytes` stores the text encoder in 4-bit. Colab's own `torchao` is older than the current `diffusers` expects and breaks `import diffusers`, so it is removed (nothing here uses it).

In [ ]:
!pip -q install optimum-quanto bitsandbytes
!pip -q uninstall -y torchao

## 2. Get the model (about 25 GB)
This model is **27 GB, which does not fit a free Google Drive (15 GB)**, so on a free account it downloads to the session's disk every time. With a Drive that has 30 GB or more free, the first run saves it and later runs skip the download.
## Model storage: reuse it from Google Drive, download it only if it is missing
The next cell defines `get_model()`. It mounts Google Drive (Colab will ask you to allow it) and keeps models in `MyDrive/colab-models/`.
- **Already in Drive:** it is used as it is, with no download.
- **Missing and it fits:** it is downloaded straight into Drive, so the next session skips the download.
- **Missing and Drive is too small** (a free Drive account has only 15 GB in total): it downloads to this session's disk instead and tells you so.
- **Say no to the Drive prompt, or set `USE_DRIVE = False`:** it simply uses the session's disk.

In [ ]:
"""Model cache for Colab: reuse a model saved in Google Drive, download it if it is missing, fall back to the local disk if Drive is too small.

Paste this cell near the top of a notebook. Then:

    MODEL_DIR = get_model("black-forest-labs/FLUX.2-klein-4B", "flux2-klein-4b",
                          allow_patterns=[...], ignore_patterns=[...])

What it does, in order:
  1. Mounts Google Drive (skipped if USE_DRIVE = False or the mount is refused) and uses  MyDrive/colab-models/<name>.
  2. If that folder already holds a COMPLETE copy (a marker file written after a finished download), it is used as it is: no download.
  3. Otherwise it asks Hugging Face how big the download is and compares that with the free space in Drive:
       - it fits (with 5% spare): download straight into Drive, so the next session skips the download;
       - it does not fit: download to Colab's local disk (/content/models/<name>) and say so. Free Drive is only 15 GB in total.
  4. A download that was cut off is resumed, not restarted (huggingface_hub keeps partial files).
Nothing is deleted. Returns the folder to pass to from_pretrained().
"""
import os, shutil, json, time

USE_DRIVE = True                                     # set False to always use the local disk
DRIVE_ROOT = "/content/drive/MyDrive/colab-models"   # where models are kept in your Drive
LOCAL_ROOT = "/content/models"                       # used when Drive is off or too small
_MARK = ".download_complete.json"


def _mount_drive():
    if not USE_DRIVE:
        return False
    if os.path.isdir("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return os.path.isdir("/content/drive/MyDrive")
    except Exception as e:                           # not in Colab, or the user said no
        print("Google Drive not available (" + type(e).__name__ + "): using the local disk")
        return False


def _remote_size(repo, allow_patterns, ignore_patterns):
    from huggingface_hub import HfApi
    import fnmatch
    total = 0
    for f in HfApi().list_repo_tree(repo, recursive=True):
        size = getattr(f, "size", None)
        if size is None:
            continue
        p = f.path
        if allow_patterns and not any(fnmatch.fnmatch(p, a) for a in allow_patterns):
            continue
        if ignore_patterns and any(fnmatch.fnmatch(p, i) for i in ignore_patterns):
            continue
        total += size
    return total


def get_model(repo, name, allow_patterns=None, ignore_patterns=None, revision=None):
    """Return a local folder holding `repo`, using Drive as a cache when it can."""
    from huggingface_hub import snapshot_download
    ignore_patterns = list(ignore_patterns or []) + [_MARK]
    drive_ok = _mount_drive()
    drive_dir = os.path.join(DRIVE_ROOT, name)
    local_dir = os.path.join(LOCAL_ROOT, name)

    for d, where in ((drive_dir, "Google Drive"), (local_dir, "this session's disk")):
        if os.path.exists(os.path.join(d, _MARK)):
            print(f"Using the saved copy in {where}: {d}  (no download)")
            return d

    need = _remote_size(repo, allow_patterns, ignore_patterns)
    print(f"{repo}: about {need / 1e9:.1f} GB to download")
    target = local_dir
    if drive_ok:
        os.makedirs(DRIVE_ROOT, exist_ok=True)
        used_by_us = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(drive_dir) for f in fs) if os.path.isdir(drive_dir) else 0
        free = shutil.disk_usage("/content/drive/MyDrive").free + used_by_us
        if need * 1.05 < free:
            target = drive_dir
            print(f"It fits in Google Drive ({free / 1e9:.1f} GB free): saving to {drive_dir} so the next session skips the download.")
        else:
            print(f"It does NOT fit in Google Drive ({free / 1e9:.1f} GB free, free accounts get 15 GB): using this session's disk, "
                  "so the next session downloads it again. Free up Drive space or use Google One to keep it.")
    os.makedirs(target, exist_ok=True)
    t = time.time()
    snapshot_download(repo, local_dir=target, allow_patterns=allow_patterns, ignore_patterns=ignore_patterns, revision=revision)
    json.dump({"repo": repo, "revision": revision, "bytes": need, "seconds": round(time.time() - t)}, open(os.path.join(target, _MARK), "w"))
    print(f"Done in {time.time() - t:.0f}s: {target}")
    return target

In [ ]:
MODEL_DIR = get_model("OpenMOSS-Team/MOSS-Image-SFT", "moss-image-sft")
print("model files in", MODEL_DIR)

## 3. Load it in low-memory form
The T4 has no native bfloat16, so everything runs in float16. Loading takes about 2.5 minutes and leaves about 11.5 GB on the GPU.

In [ ]:
import time, torch
from diffusers import DiffusionPipeline, QuantoConfig
from diffusers.quantizers import PipelineQuantizationConfig
from transformers import BitsAndBytesConfig

DT = torch.float16
quant = PipelineQuantizationConfig(quant_mapping={
    "transformer": QuantoConfig(weights_dtype="int8"),
    "text_encoder": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DT),
})
t = time.time()
pipe = DiffusionPipeline.from_pretrained(
    MODEL_DIR, custom_pipeline="pipeline_moss_image", trust_remote_code=True,
    torch_dtype={"default": DT, "vae": torch.float32}, quantization_config=quant)
pipe.scheduler = type(pipe.scheduler).from_config(pipe.scheduler.config)
pipe.to("cuda")
print(f"loaded in {time.time() - t:.0f}s, {torch.cuda.memory_allocated() / 1e9:.1f} GB on the GPU")

## 4. Make a picture
`size` must be a multiple of 16. **768** is the safe choice on a free T4 (about 2 minutes, 12.9 GB peak). **1024** works but needs 14 GB at peak and takes about 4 minutes. `steps=30` is what was tested here, on flat mascot art; the model card's default is 50, which I did not try, so raise it if a picture looks rough.

In [ ]:
from IPython.display import display

NEGATIVE = "realistic, photo, 3d, shading, gradient, watermark, extra limbs"

def draw(prompt, size=768, steps=30, seed=42, negative=NEGATIVE):
    t = time.time(); torch.cuda.reset_peak_memory_stats()
    img = pipe(prompt, negative_prompt=negative, height=size, width=size, num_inference_steps=steps,
               guidance_scale=3.0, generator=torch.Generator("cpu").manual_seed(seed)).images[0]
    print(f"{time.time() - t:.0f}s, peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    return img

MARKO = ("a baby markhor mountain goat mascot standing upright on two legs, full body, front view, round fluffy turquoise-teal fur, "
         "flat cream face mask with a tiny nose and a small cat-like smile, big round dark navy eyes with white highlights, cream inner ears "
         "pointing sideways, two tall twisted corkscrew spiral saffron-orange horns, small beard tuft, cream round belly, little saffron hooves, "
         "thick dark navy outline, solid flat colours, plain white background")

img = draw("Marko, waving one hand. A " + MARKO)
display(img)
img.save("marko_wave.png")

## 5. Text inside the picture
This is what MOSS-Image is built for. English text was spelled correctly in both of my tests. Urdu script looked convincing but only the first word of my test sign was right, so check Urdu spelling yourself.

In [ ]:
img = draw('A baby markhor mascot named Marko holding up a cream sign with large bold English text written on it: "READ URDU WITH MARKO". '
           'full body, front view, round fluffy turquoise-teal fur, cream face, big navy eyes, two tall saffron corkscrew horns, '
           'thick dark navy outline, solid flat colours, plain white background')
display(img)
img.save("marko_sign_english.png")

## 6. Your own prompt
Change the text and run the cell. Prompts can be plain sentences, tag lists, or a JSON layout that places text and objects (see the [MOSS-Image user guide](https://github.com/OpenMOSS/MOSS-Image/blob/main/USERGUIDE.md)).

In [ ]:
PROMPT = "A cozy village school classroom with a chalkboard that says \"WELCOME\", flat vector illustration, warm colours"
img = draw(PROMPT, size=768, steps=30, seed=7)
display(img)
img.save("mine.png")